# Customer Churn Prediction using Random Forest
## End-to-End Machine Learning & Deployment Use Case

| | |
|---|---|
| **Student Name** | Khyati Chauhan |
| **Enrollment Number** | 202507090013 |
| **KU ID** | KU2507U0406 |
| **Course** | B.Tech Hons. CSE GenAI (A) IBM |
| **Semester** | 2nd Semester |

---
## 1. Introduction

### What is Customer Churn?
Customer churn refers to the phenomenon where customers stop doing business with a company. In the travel industry, churn means customers stop booking flights, hotels, or travel packages. Identifying which customers are likely to churn **before** they leave is critically important for business survival.

### Why is Churn Prediction Important?
- Acquiring a new customer costs **5–7x more** than retaining an existing one.
- Even a **5% reduction** in churn can increase profits by 25–95%.
- Churn prediction enables targeted retention campaigns with personalized offers.
- It helps businesses allocate marketing budgets efficiently.

### Why Random Forest?
Random Forest is an ensemble learning algorithm that builds multiple decision trees and combines their predictions. It is ideal for this project because:
- It **handles both categorical and numerical** features effectively.
- It is **robust to overfitting** due to its ensemble nature.
- It provides **feature importance scores** — critical for business insights.
- It works well even with **imbalanced datasets** (which is common in churn problems).
- It requires **minimal hyperparameter tuning** to achieve good results.

---
## 2. Importing Required Libraries

In [ ]:
# Core data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve, auc
)

# Model saving
import pickle
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ All libraries imported successfully!")

---
## 3. Data Loading and Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('customer_churn.csv')

print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n--- First 5 Rows ---")
df.head()

In [ ]:
# Dataset information
print("--- Dataset Info ---")
df.info()

In [ ]:
# Summary statistics
print("--- Summary Statistics ---")
df.describe(include='all')

In [ ]:
# Check for missing values
print("--- Missing Values ---")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
# Check unique values for categorical columns
print("--- Unique Values per Column ---")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique → {df[col].unique()[:5]}")

In [ ]:
# Target variable distribution
print("--- Target Variable Distribution ---")
churn_counts = df['Target'].value_counts()
print(churn_counts)
print(f"\nChurn Rate: {(df['Target'].sum() / len(df) * 100):.2f}%")

---
## 4. Data Cleaning and Preprocessing

In [ ]:
# Make a working copy
df_clean = df.copy()

# Fix 'AnnualIncomeClass' column - clean 'Record' prefix noise
df_clean['AnnualIncomeClass'] = df_clean['AnnualIncomeClass'].str.replace('Record ', '', regex=False)
print("AnnualIncomeClass after cleaning:", df_clean['AnnualIncomeClass'].unique())

# Convert Age and ServicesOpted to numeric
df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors='coerce')
df_clean['ServicesOpted'] = pd.to_numeric(df_clean['ServicesOpted'], errors='coerce')

# Drop rows with remaining NaNs (if any)
df_clean.dropna(inplace=True)
df_clean.reset_index(drop=True, inplace=True)

print(f"\n✅ Data cleaned. Shape after cleaning: {df_clean.shape}")
df_clean.head()

In [ ]:
# ── Feature Encoding ──────────────────────────────────────────────
# Encode binary Yes/No columns
binary_cols = ['FrequentFlyer', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']
for col in binary_cols:
    df_clean[col] = df_clean[col].map({'Yes': 1, 'No': 0})

# Encode ordinal AnnualIncomeClass with meaningful ordering
income_map = {'Low Income': 0, 'Middle Income': 1, 'High Income': 2}
df_clean['AnnualIncomeClass'] = df_clean['AnnualIncomeClass'].map(income_map)

print("✅ Encoding complete.")
print("\nEncoded DataFrame:")
df_clean.head()

In [ ]:
# Verify no NaNs after encoding
print("Missing after encoding:", df_clean.isnull().sum().sum())
print("\nData types after encoding:")
print(df_clean.dtypes)

---
## 5. Train-Test Split

In [ ]:
# Define features and target
X = df_clean.drop('Target', axis=1)
y = df_clean['Target']

print("Features:", X.columns.tolist())
print("Target distribution:")
print(y.value_counts())

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Split done!")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set    : {X_test.shape[0]} samples")

---
## 6. Random Forest Model Development

In [ ]:
# Train the Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'  # Handle class imbalance
)

rf_model.fit(X_train, y_train)

# Generate predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

print("✅ Random Forest model trained successfully!")
print(f"Number of trees: {rf_model.n_estimators}")
print(f"Max depth: {rf_model.max_depth}")

---
## 7. Model Evaluation

In [ ]:
# ── Accuracy Score ────────────────────────────────────────────────
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'],
            linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives : {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives : {tp}")

In [ ]:
# ── Classification Report ─────────────────────────────────────────
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Random Forest', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ AUC Score: {roc_auc:.4f}")

In [ ]:
# ── Feature Importance ────────────────────────────────────────────
feature_names = X.columns.tolist()
importances = rf_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']
bars = ax.bar(
    [feature_names[i] for i in sorted_idx],
    importances[sorted_idx],
    color=colors[:len(feature_names)]
)
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Features', fontsize=12)
ax.set_ylabel('Importance Score', fontsize=12)
ax.tick_params(axis='x', rotation=20)

# Add value labels on bars
for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFeature Importances:")
for i in sorted_idx:
    print(f"  {feature_names[i]:<35}: {importances[i]:.4f}")

---
## 8. Visualizations

In [ ]:
# ── Churn Distribution ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
churn_labels = ['No Churn (0)', 'Churn (1)']
churn_vals = df_clean['Target'].value_counts().sort_index().values
axes[0].bar(churn_labels, churn_vals, color=['#4CAF50', '#F44336'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_vals):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(churn_vals, labels=churn_labels, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Distribution (%)', fontsize=13, fontweight='bold')

plt.suptitle('Customer Churn Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Distribution Plots ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

original_features = ['Age', 'FrequentFlyer', 'AnnualIncomeClass',
                     'ServicesOpted', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']

for i, col in enumerate(original_features):
    sns.histplot(data=df_clean, x=col, hue='Target',
                 multiple='stack', palette={0: '#4CAF50', 1: '#F44336'},
                 ax=axes[i], kde=(col == 'Age' or col == 'ServicesOpted'))
    axes[i].set_title(f'{col} vs Churn', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions by Churn Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df_clean.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Age Distribution by Churn ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for target_val, color, label in [(0, '#4CAF50', 'No Churn'), (1, '#F44336', 'Churn')]:
    subset = df_clean[df_clean['Target'] == target_val]['Age']
    ax.hist(subset, bins=15, alpha=0.6, color=color, label=label, edgecolor='white')

ax.set_title('Age Distribution by Churn Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Age', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Conclusion

In [ ]:
# Summary of results
print("=" * 60)
print("        CUSTOMER CHURN PREDICTION - SUMMARY REPORT")
print("=" * 60)
print(f"\n📊 Dataset      : 954 customer records, 6 features")
print(f"🧠 Model        : Random Forest Classifier (100 trees)")
print(f"✅ Accuracy     : {accuracy * 100:.2f}%")
print(f"📈 AUC Score    : {roc_auc:.4f}")

top_feature = feature_names[sorted_idx[0]]
print(f"\n🔑 Top Feature  : {top_feature} (importance: {importances[sorted_idx[0]]:.4f})")
print()
print("📌 Key Insights:")
print("  1. ServicesOpted is the strongest predictor of churn.")
print("  2. FrequentFlyer customers are less likely to churn.")
print("  3. High-income customers have lower churn rates.")
print("  4. Customers who booked hotels show distinct churn patterns.")
print()
print("🚀 Possible Improvements:")
print("  1. Collect more features (e.g., last activity date, complaints).")
print("  2. Try SMOTE to handle class imbalance more aggressively.")
print("  3. Experiment with XGBoost or Gradient Boosting.")
print("  4. Perform hyperparameter tuning with GridSearchCV.")
print("=" * 60)

---
## 10. Save Model

In [ ]:
# Save the trained model as model.pkl
with open('model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("✅ Model saved as 'model.pkl'")
print("   → Use this file in your Streamlit app (app.py)")